# Chapter 05 · Notebook 02: Training the layer

[Chapter guide](README.md) · [Previous notebook](01_linear_layer.ipynb) · [Course index](../README.md)

We already made predictions and measured loss with a linear layer. Now connect its parameters to the gradients and SGD updates we learned earlier. Follow one lesson at a time with the mentoring conversation.

## Contents

1. [Calculate gradients](#gradients)
2. [Update the layer with SGD](#update)
3. [Put the familiar sequence together](#sequence)
4. [Repeat the training step](#repeat)
5. [Use PyTorch's mean squared loss](#mse)

Run Setup first in a fresh kernel. Each notebook owns its variables; variables from notebook 01 are not prerequisites here.

## Setup: recreate our familiar starting point

Use four examples, one input feature each, and matching target shapes. Explicitly set weight 1 and bias 0. The starting loss should be 30. This cell does not calculate gradients or train the layer.

In [ ]:
import torch

# Each row contains one example with one input value.
x = torch.tensor([[0.], [1.], [2.], [3.]])
y = torch.tensor([[2.], [5.], [8.], [11.]])
model = torch.nn.Linear(1, 1)

# Set a known starting point without tracking initialization operations.
with torch.no_grad():
    model.weight.fill_(1.)
    model.bias.fill_(0.)

with torch.no_grad():
    starting_loss = ((model(x) - y) ** 2).mean().item()
print("Starting weight:", model.weight.item(), "Bias:", model.bias.item())
print("Starting loss:", starting_loss)


<a id="gradients"></a>
## Lesson 01: calculate gradients without updating parameters

`loss.backward()` calculates gradients and stores them on the layer's weight and bias. Access them through `model.weight.grad` and `model.bias.grad`, just as we accessed `w.grad` and `b.grad` before.

Clear old gradients first, then make new predictions and a new loss. Clearing gradients and recomputing the forward pass makes this cell safe to rerun before the update lesson. The cell calculates gradients at the **current** parameters.

**Before running:** will weight and bias change? Which gradients do you expect at weight 1 and bias 0?

**My prediction:** _write here_


In [ ]:
# Clear gradients from any previous backward call; parameter values stay unchanged.
model.zero_grad(set_to_none=True)

# Make a fresh forward pass so backward has a fresh calculation to follow.
predictions = model(x)
loss = ((predictions - y) ** 2).mean()
print("Parameters before backward:", model.weight.item(), model.bias.item())

loss.backward()  # Calculate gradients; this does not update parameters.

print("Weight gradient:", model.weight.grad.item())
print("Bias gradient:", model.bias.grad.item())
print("Parameters after backward:", model.weight.item(), model.bias.item())


**My observation and explanation:** _write here_

<a id="update"></a>
## Lesson 02: give the layer's parameters to SGD

Earlier we passed `[w, b]` to the optimizer. `model.parameters()` gives it the parameters owned by this layer: its weight and bias. They are the existing parameters, so the optimizer updates the values used by `model(x)`.

The previous lesson has already calculated gradients. Do **not** clear them between that backward call and this update. Creating the optimizer does not change parameters; `optimizer.step()` applies the update. Here we use plain SGD with learning rate 0.05, momentum 0, and weight decay 0.

**Before running:** use `new value = old value - learning rate × gradient` to predict the new weight and bias.

**My prediction:** _write here_


In [ ]:
# Register the layer's existing weight and bias with the optimizer.
optimizer = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0, weight_decay=0)
print("Parameters before step:", model.weight.item(), model.bias.item())

optimizer.step()  # Use the gradients calculated in Lesson 01.

# Evaluate the updated model without recording gradient calculations.
with torch.no_grad():
    updated_predictions = model(x)
    updated_loss = ((updated_predictions - y) ** 2).mean().item()

print("Parameters after step:", model.weight.item(), model.bias.item())
print("Updated predictions:", updated_predictions.flatten().tolist())
print("Updated loss:", updated_loss)


**My observation and explanation:** _write here_

Run this update cell once after Lesson 01. Repeating `step()` alone would reuse old gradients, rather than calculate gradients at the new parameters. The next lesson shows the complete order.

<a id="sequence"></a>
## Lesson 03: put the familiar sequence together

Reset weight and bias to our starting point so this complete training step can be compared with Lesson 02. This deliberate reset is for the demonstration; ordinary training continues from the previous update.

Create the optimizer once before repeated training steps. Each step clears gradients, makes fresh predictions, measures loss, calculates gradients, and updates parameters. After updating, make fresh predictions to measure the new loss: the earlier `loss` still records the pre-update calculation.

**Before running:** which line calculates gradients, and which line changes the parameters?

**My answer:** _write here_


In [ ]:
# Reset only for this comparison with our first manual training step.
with torch.no_grad():
    model.weight.fill_(1.)
    model.bias.fill_(0.)
optimizer = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0, weight_decay=0)

optimizer.zero_grad(set_to_none=True)  # Clear old gradients.
predictions = model(x)               # Make fresh predictions.
loss = ((predictions - y) ** 2).mean()  # Measure pre-update error.
loss.backward()                      # Calculate gradients.
optimizer.step()                     # Update weight and bias.

# The old loss is not automatically recalculated after the update.
with torch.no_grad():
    loss_after = ((model(x) - y) ** 2).mean().item()

print("Loss before update:", loss.item())
print("Weight:", model.weight.item(), "Bias:", model.bias.item())
print("Loss after update:", loss_after)


**My observation and explanation:** _write here_

We have changed how parameters are stored and passed to SGD. The prediction rule, mean squared loss, gradients, and plain SGD update are the same as before.


<a id="repeat"></a>
## Lesson 04: repeat the training step ten times

Each pass through the loop is the same complete training step from Lesson 03. Clear gradients, make fresh predictions, calculate loss, calculate gradients, and update the layer. Each step uses all four examples: one full-batch update is one epoch for this experiment.

Reset weight to 1 and bias to 0 before the loop so the result can be compared with our earlier ten manual updates. The reset also makes this experiment repeatable. Create the optimizer once before the loop; keep using it for every update.

The printed loss is measured **after** each update with a fresh forward pass. The weight gradient shown was calculated **before** that update. We print the gradient before calling backward again on the next step.

**Before running:** should the second update calculate the same gradient as the first? Why?

**My prediction:** _write here_


In [ ]:
# Reset for a fair comparison with our earlier ten-update experiment.
with torch.no_grad():
    model.weight.fill_(1.)
    model.bias.fill_(0.)

# Create the optimizer once; all ten steps update this same layer.
optimizer = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0, weight_decay=0)

for step in range(1, 11):
    optimizer.zero_grad(set_to_none=True)  # Clear the previous step's gradients.
    predictions = model(x)                # Predict with the current parameters.
    loss = ((predictions - y) ** 2).mean()  # Measure the current training error.
    loss.backward()                       # Calculate fresh gradients.
    weight_gradient = model.weight.grad.item()
    optimizer.step()                      # Apply one parameter update.

    # Recalculate loss after the update; the earlier loss is the pre-update value.
    with torch.no_grad():
        loss_after = ((model(x) - y) ** 2).mean().item()

    print(
        f"Step {step:2d} | grad_w={weight_gradient:8.4f} | "
        f"weight={model.weight.item():.4f} | bias={model.bias.item():.4f} | "
        f"loss={loss_after:.4f}"
    )


**My observation and explanation:** _write here_

Compare with Chapter 01's manual update: the first update should still give weight 2, bias 0.5, and loss 10.25. Moving parameter storage into a layer does not change plain SGD's update rule. The second gradient is calculated at the new parameters, rather than reused from the first update.

A decreasing loss in this run is an observation about these data and settings. It is not a guarantee for every learning rate, optimizer, or training dataset.


<a id="mse"></a>
## Lesson 05: use PyTorch's mean squared loss

Our handwritten loss is `((predictions - y) ** 2).mean()`. `torch.nn.MSELoss()` performs the same mean squared error calculation by default. MSE means **mean squared error**.

Create the loss function once, then call it with predictions and targets: `loss_fn(predictions, y)`. The loss function has no learned weight or bias here; it only measures error. Predictions and targets must have matching shape `[4, 1]` in this experiment.

The comparison below uses the current trained parameters from Lesson 04. It only evaluates loss, so it uses `torch.no_grad()`. When calculating loss for training, make the forward pass and loss calculation outside `no_grad()` so `loss.backward()` can calculate parameter gradients.

**Before running:** should the two loss values agree? Does using this function update the parameters?

**My prediction:** _write here_


In [ ]:
# Create the mean squared error loss function; default reduction is the mean.
loss_fn = torch.nn.MSELoss()

# Compare both calculations on exactly the same predictions and targets.
# Evaluation does not require tracking gradients.
with torch.no_grad():
    predictions = model(x)
    handwritten_loss = ((predictions - y) ** 2).mean()
    pytorch_loss = loss_fn(predictions, y)

print("Handwritten loss:", handwritten_loss.item())
print("PyTorch MSE loss:", pytorch_loss.item())
print("Loss values agree:", torch.allclose(handwritten_loss, pytorch_loss))
print("Parameters still:", model.weight.item(), model.bias.item())


**My observation and explanation:** _write here_

For a later training step, replace only the loss calculation with `loss = loss_fn(predictions, y)`. Clearing gradients, backward, and the optimizer update keep their familiar roles.


## Documentation

- [PyTorch: Tensor.backward](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.backward.html)
- [PyTorch: Module.parameters](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.parameters)
- [PyTorch: SGD](https://docs.pytorch.org/docs/stable/generated/torch.optim.SGD.html)

- [PyTorch: MSELoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.MSELoss.html)
